<a href="https://colab.research.google.com/github/mrhelek/proyekakhirNLP/blob/main/skenario_2_Lora_(dataset%2BLLM%2BLora).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SKENARIO 2: LoRA FINE-TUNING (TinyLlama + Alpaca) & EVALUASI
# ============================================================
import os
os.environ["PEFT_USE_TORCHAO"] = "0"

# ---------- 1. Install semua dependensi (sama seperti BitDistiller) ----------
print("🔧 Menginstall dependensi yang diperlukan...")
!git clone https://github.com/BrownianNotion/BitDistiller.git 2>/dev/null || echo "Repo sudah ada"
%cd /content/BitDistiller
!pip install -q -r requirements.txt
!pip install -q protobuf sentencepiece

# ---------- 2. Import library ----------
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset
from huggingface_hub import snapshot_download
import json
import pandas as pd
import subprocess
import re

# ---------- 3. Model base ----------
base_model_dir = "/content/models/tinyllama_base"
if not os.path.exists(base_model_dir):
    print("📥 Mengunduh model base TinyLlama...")
    snapshot_download(
        repo_id="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
        local_dir=base_model_dir,
        ignore_patterns=["*.bin", "*.h5"]
    )
print("✅ Model base siap.\n")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"🚀 Menggunakan device: {device}")

tokenizer = AutoTokenizer.from_pretrained(base_model_dir)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    base_model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# ---------- 4. Dataset Alpaca (5.000 sampel) ----------
print("\n📚 Memuat dataset Alpaca (5.000 sampel)...")
dataset = load_dataset("tatsu-lab/alpaca", split="train")
dataset = dataset.select(range(5000))

def format_instruction(example):
    instr = example["instruction"]
    inp = example["input"]
    out = example["output"]
    if inp:
        text = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n{out}"
    else:
        text = f"### Instruction:\n{instr}\n\n### Response:\n{out}"
    return {"text": text}

dataset = dataset.map(format_instruction)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

# ---------- 5. Konfigurasi LoRA ----------
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ---------- 6. Training ----------
output_dir = "/content/lora_tinyllama_alpaca"
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("\n🔥 Memulai LoRA fine-tuning (1 epoch, 5000 sampel)...")
trainer.train()
model.save_pretrained(output_dir)
print(f"✅ LoRA adapter disimpan di {output_dir}")

# ---------- 7. Merge LoRA dengan base model ----------
print("\n🔀 Menggabungkan adapter LoRA ke model base...")
base_model_fresh = AutoModelForCausalLM.from_pretrained(
    base_model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer_fresh = AutoTokenizer.from_pretrained(base_model_dir)
tokenizer_fresh.pad_token = tokenizer_fresh.eos_token

peft_model = PeftModel.from_pretrained(base_model_fresh, output_dir)
merged_model = peft_model.merge_and_unload()
merged_model_dir = "/content/models/tinyllama_lora_merged"
merged_model.save_pretrained(merged_model_dir)
tokenizer_fresh.save_pretrained(merged_model_dir)
print(f"✅ Model hasil merge disimpan di {merged_model_dir}")

# ---------- 8. Evaluasi dengan script BitDistiller ----------
eval_dir = "/content/BitDistiller/test/general"
os.chdir(eval_dir)

# Perplexity (WikiText-2)
print("\n" + "=" * 60)
print("📊 Menghitung WikiText-2 Perplexity (model LoRA)...")
print("=" * 60)
cmd_ppl = [
    "python", "wiki_ppl.py",
    "--model", merged_model_dir,
    "--quant_type", "fp",
    "--bits", "16",
    "--group_size", "128"
]
proc_ppl = subprocess.run(cmd_ppl, capture_output=True, text=True)
print(proc_ppl.stdout)
if proc_ppl.stderr:
    print("STDERR:", proc_ppl.stderr)

ppl_match = re.search(r"ppl:\s*\n?\s*([\d.]+)", proc_ppl.stdout + proc_ppl.stderr)
if ppl_match:
    ppl_lora = round(float(ppl_match.group(1)), 4)
    print(f"\n✅ Perplexity (WikiText-2): {ppl_lora}")
else:
    ppl_lora = None
    print("\n⚠️ Gagal mengekstrak PPL.")

# Zero-shot accuracy (HellaSwag)
print("\n" + "=" * 60)
print("📊 Menghitung Zero-shot Accuracy (HellaSwag) untuk model LoRA...")
print("=" * 60)
cmd_qa = [
    "python", "llm_eval.py",
    "--model", merged_model_dir,
    "--eval_tasks", "hellaswag",
    "--test_set",
    "--bits", "16",
    "--group_size", "128",
    "--quant_type", "fp",
    "--num_fewshot", "0",
    "--batch_size", "2"
]
proc_qa = subprocess.run(cmd_qa, capture_output=True, text=True)
print(proc_qa.stdout)
if proc_qa.stderr:
    print("STDERR:", proc_qa.stderr)

acc_match = re.search(r"'acc_norm':\s*([\d.]+)", proc_qa.stdout + proc_qa.stderr)
if not acc_match:
    acc_match = re.search(r"'acc':\s*([\d.]+)", proc_qa.stdout + proc_qa.stderr)
if acc_match:
    acc_lora = round(float(acc_match.group(1)) * 100, 2)
    print(f"\n✅ HellaSwag acc_norm: {acc_lora}%")
else:
    acc_lora = None
    print("\n⚠️ Gagal mengekstrak akurasi.")

# ---------- 9. Simpan hasil ----------
results_lora = {
    "model": "TinyLlama-1.1B + LoRA (Alpaca 5k, 1 epoch)",
    "wikitext2_ppl": ppl_lora,
    "hellaswag_acc_norm": acc_lora
}
os.makedirs("/content/results", exist_ok=True)
with open("/content/results/scenario2_lora.json", "w") as f:
    json.dump(results_lora, f, indent=2)

df = pd.DataFrame([results_lora]).T
df.columns = ["Value"]
print("\n" + "=" * 60)
print("📋 HASIL SKENARIO 2 (LoRA)")
print(df.to_markdown())
print("=" * 60)
print("💾 Hasil disimpan di /content/results/scenario2_lora.json")

🔧 Menginstall dependensi yang diperlukan...
Repo sudah ada
/content/BitDistiller
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 61.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 138.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.9 MB/s eta 0:00:00
   ━━

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

✅ Model base siap.

🚀 Menggunakan device: cuda:0

📚 Memuat dataset Alpaca (5.000 sampel)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.10229075496156657

🔥 Memulai LoRA fine-tuning (1 epoch, 5000 sampel)...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False)
  warnings.warn(


Step,Training Loss
50,1.471800
100,1.255200
150,1.253000
200,1.253400
250,1.256200
300,1.276200
350,1.256400
400,1.263200
450,1.264100
500,1.228900


Output streaming akan dipotong hingga 5000 baris terakhir.
100%|██████████| 40145/40145 [11:12<00:00, 59.66it/s]


✅ HellaSwag acc_norm: 59.5%

📋 HASIL SKENARIO 2 (LoRA)
|                    | Value                                      |
|:-------------------|:-------------------------------------------|
| model              | TinyLlama-1.1B + LoRA (Alpaca 5k, 1 epoch) |
| wikitext2_ppl      | 7.9962                                     |
| hellaswag_acc_norm | 59.5                                       |
💾 Hasil disimpan di /content/results/scenario2_lora.json
